SETUP

In [23]:
import pandas as pd
import numpy as np

🟢 NÍVEL 1 — Exploração Inicial

In [24]:
# 1. Carregando os dados
df = pd.read_csv("livros.csv", sep=";")
df.head()

,titulo,autor,isbn,paginas,ano
0,Orçamento sem falhas,Nath Finanças,9.786556e+12,128,2021.0
1,Minha Sombria Vanessa,Kate Elizabeth Russell,9.788551e+12,432,2020.0
2,Recursão,Blake Crouch,9.788551e+12,320,2020.0
3,"M, o Filho do Século",Antonio Scurati,9.788551e+12,816,2020.0
4,Oblivion Song: Entre Dois Mundos,Robert Kirkman,9.788551e+12,136,2020.0


In [25]:
# 2. Dando uma olhada na estrutura, nulos e tipos
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 11967 entries, 0 to 11966
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   titulo   11967 non-null  str    
 1   autor    11955 non-null  str    
 2   isbn     11209 non-null  float64
 3   paginas  11967 non-null  int64  
 4   ano      11966 non-null  float64
dtypes: float64(2), int64(1), str(2)
memory usage: 886.9 KB


,isbn,paginas,ano
count,1.120900e+04,11967.000000,11966.000000
mean,9.785215e+12,276.646277,2008.340966
std,1.698072e+11,176.030445,64.569923
min,3.680000e+02,0.000000,0.000000
25%,9.788533e+12,168.000000,2007.000000
50%,9.788551e+12,256.000000,2012.000000
75%,9.788579e+12,348.000000,2016.000000
max,9.999097e+12,4606.000000,2021.000000


In [26]:
# 3. Contando os valores nulos em cada coluna
df.isnull().sum()

titulo       0
autor       12
isbn       758
paginas      0
ano          1
dtype: int64

In [27]:
# 4. Filtrando pra ver se tem livro com 0 páginas
livros_zerados = df[df["paginas"] == 0]

print(f"Total de livros com 0 páginas: {len(livros_zerados)}")
livros_zerados.head()

Total de livros com 0 páginas: 42


,titulo,autor,isbn,paginas,ano
15,DUPLICADO Este é o Mar,Mariana Enriquez,9.788551e+12,0,2020.0
142,O Caso Da Mansão Deboën,Edgar Cantero,9.788551e+12,0,2019.0
155,Mundo Em Caos (Vol. 1),Patrick Ness,9.788551e+12,0,2019.0
179,Matadouro-Cinco,Kurt Vonnegut,9.788551e+12,0,2019.0
204,Coleção Josh Malerman,NaN,9.788551e+12,0,2019.0


In [28]:
# 5. Quantidade de livros por ano (ordem cronológica)
df["ano"].value_counts().sort_index()

ano
0.0        12
1193.0      1
1862.0      1
1884.0      1
1931.0      1
         ... 
2017.0    860
2018.0    655
2019.0    737
2020.0    591
2021.0      5
Name: count, Length: 66, dtype: int64

🔵 Nível 2 — Transformação e Limpeza

In [29]:
# 1. Classificando pelo tamanho usando lambda pra aplicar as 3 condições direto
df["faixa_paginas"] = df["paginas"].apply(
    lambda x: "Curto" if x < 150 else ("Médio" if x <= 350 else "Longo")
)
df["faixa_paginas"].value_counts()

faixa_paginas
Médio    6633
Longo    2947
Curto    2387
Name: count, dtype: int64

In [30]:
# 2. Filtrando fora os livros com 0 páginas e criando o df_limpo
tamanho_original = len(df)
df_limpo = df[df["paginas"] > 0].copy()

print(f"Registros removidos: {tamanho_original - len(df_limpo)}")

Registros removidos: 42


In [31]:
# 3. Tapando os buracos do ano com a mediana e arrumando o tipo pra int
mediana_ano = df_limpo["ano"].median()
df_limpo["ano"] = df_limpo["ano"].fillna(mediana_ano).astype(int)

In [32]:
# 4. Extraindo a década.
# A divisão inteira por 10 tira o último número, depois multiplica de novo
df_limpo["decada"] = (df_limpo["ano"] // 10) * 10
df_limpo[["ano", "decada"]].head()

,ano,decada
0,2021,2020
1,2020,2020
2,2020,2020
3,2020,2020
4,2020,2020


🟡 Nível 3 — Análise Avançada

In [33]:
# 1. Média de páginas agrupada por década
df_limpo.groupby("decada")["paginas"].mean().sort_index().round(1)

decada
0       168.0
1190    278.0
1860    160.0
1880    248.0
1930    151.0
1940    224.0
1950    235.8
1960    313.7
1970    172.3
1980    246.4
1990    262.7
2000    240.7
2010    293.7
2020    336.7
Name: paginas, dtype: float64

In [34]:
# 2. Top 10 autores que mais publicaram
df_limpo["autor"].value_counts().head(10)

autor
Clarice Lispector    104
Rocco                 88
Rick Riordan          84
Meg Cabot             73
Augusto Cury          71
Paulo Coelho          70
J.K. Rowling          66
Planeta do Brasil     62
Lucasfilm Ltd.        59
Cassandra Clare       57
Name: count, dtype: int64

In [35]:
# 3. Distribuição do tamanho dos livros publicados de 2011 pra frente
df_limpo[df_limpo["ano"] > 2010]["faixa_paginas"].value_counts()

faixa_paginas
Médio    4010
Longo    1961
Curto     970
Name: count, dtype: int64

In [36]:
# 4. Jogando o resultado final pro Excel
df_limpo.to_excel("livros_analisados.xlsx", index=False)